[Source](https://docs.nvidia.com/bionemo-framework/1.10/notebooks/MolMIM_GenerativeAI_local_inference_with_examples.html)

In [ ]:
%%capture --no-display --no-stderr cell_output

import os
import itertools
import warnings
import logging
import random

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from tqdm import tqdm
import numpy as np

from bionemo.utils.hydra import load_model_config
from bionemo.model.molecule.molmim.infer import MolMIMInference

random.seed(42)
np.random.seed(42)

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

logging.basicConfig(level=logging.INFO)
logging.getLogger("nemo_logger").setLevel(logging.ERROR)

In [ ]:
bionemo_home = "/workspace/bionemo"
os.environ['BIONEMO_HOME'] = bionemo_home
os.chdir(bionemo_home)

In [ ]:
# %%capture --no-display cell_output
# !python download_artifacts.py --model_dir ${BIONEMO_HOME}/models --models molmim_70m_24_3

In [ ]:
%%capture --no-display --no-stderr cell_output

# Load pre-trained model checkpoints
# checkpoint_path = f"{bionemo_home}/models/molecule/molmim/molmim_70m_24_3.nemo"

checkpoint_path = "/workspace/bionemo/data/models/MolMIM_small_experts_1_max_steps_100.nemo"
# Load starting config for MolMIM inference
cfg = load_model_config(config_name="molmim_infer.yaml", config_path=f"{bionemo_home}/examples/tests/conf/")

# Point YAML configuration file to the location of the desired checkpoints
cfg.model.downstream_task.restore_from_path = checkpoint_path
#cfg.model.encoder.hidden_steps = 2

# Create model object based on desired configuration
model = MolMIMInference(cfg, interactive=True)

In [ ]:
expert_smiles_path = "/workspace/bionemo/data/data_experts_1.csv"
expert_smiles_filename = expert_smiles_path.split("/")[-1]
expert_smiles = pd.read_csv(expert_smiles_path)['smiles'].to_list()

# RDKit's MolFromSmiles() function displays molecule from the SMILES string
m1 = Chem.MolFromSmiles(expert_smiles[0])
m2 = Chem.MolFromSmiles(expert_smiles[1])
Draw.MolsToGridImage((m1, m2), legends=["Smiles 1", "Smiles 2"], subImgSize=(300, 200))

In [ ]:
# obtaining the hidden state representations for input SMILES
hidden_states, pad_masks = model.seq_to_hiddens(expert_smiles)
hidden_states.shape, pad_masks.shape

In [ ]:
embedding = model.seq_to_embeddings(expert_smiles)
embedding.shape

In [ ]:
# Obtaining SMILES chemical representation from a hidden state
inferred_smis = model.hiddens_to_seq(hidden_states, pad_masks)

inf_1 = Chem.MolFromSmiles(inferred_smis[0])
inf_2 = Chem.MolFromSmiles(inferred_smis[1])

print(f"Where inferred smiles valid? Smiles 1: {inf_1}, Smiles 2: {inf_2}")
print("Compund 1:", expert_smiles[0] == inferred_smis[0])
print("Compund 2:", expert_smiles[1] == inferred_smis[1])
# Draw.MolsToGridImage((inf_1, inf_2), legends=["Inferred Compound 1", "Inferred Compound 2"], subImgSize=(350, 350))

In [ ]:
def chem_sample(
        reference_smiles: list[str],
        num_samples: int = 100,
        scaled_radius: float = 1.0,
        sampling_method: str = "beam-search-perturbate",
        **sampler_kwargs
) -> list:
    # PART 1: SAMPLING
    # 1A: Set Sampling Arguments
    default_sampler_kwargs = {"beam_size": 3, "keep_only_best_tokens": True, "return_scores": False}
    sampler_kwargs = {**default_sampler_kwargs, **sampler_kwargs}  # Override defaults with user-provided kwargs

    # 1B: Execute sampling
    population_samples = model.sample(
        seqs=reference_smiles,
        num_samples=num_samples,
        scaled_radius=scaled_radius,
        sampling_method=sampling_method,
        **sampler_kwargs
    )

    # PART 2: FILTERING
    uniq_canonical_smiles = []
    # Loop through each seed molecule
    for smis_samples, original in zip(population_samples, reference_smiles):
        # 2A: Gather unique strings (remove duplicates and starting string)
        smis_samples = set(smis_samples) - {original}

        # 2B: Validate generated molecules
        valid_molecules = []
        for smis in smis_samples:
            mol = Chem.MolFromSmiles(smis)
            if mol:
                valid_molecules.append(Chem.MolToSmiles(mol, True))
        uniq_canonical_smiles.append(valid_molecules)
    return uniq_canonical_smiles  # List (len = seed compounds) of lists (len = generated compounds per seed)

### Sampling methods:
- "greedy-perturbate": Sample the best sequence for each of our perturbed hiddens, using greedy-search (per token)
to find the best result.
- "topkp-perturbate": Sample the best sequence for each of our perturbed hiddens, using `topkp-sampling` to
find the best result.
- "beam-search-perturbate": Sample the best sequence for each of our perturbed hiddens,
using beam-search to find the best result.
 - "beam-search-single-sample": Sample the top num_samples sequences using beam-search (rather than single best) given our
single perturbed hidden.
- "beam-search-perturbate-sample": Sample the top beam_size (default 5) sequences using beam-search
for each of our `num_samples` purturbed hiddens. This will return a `beam_size * num_samples` set of results.

In [ ]:
sampling_methods = [
    'greedy-perturbate',
    'topkp-perturbate',
    'beam-search-perturbate',
    'beam-search-single-sample'
    'beam-search-perturbate-sample'
]

## Define sampling arguments

In [ ]:
sampling_method: str = "beam-search-perturbate"

In [ ]:
model.default_sampling_kwargs[sampling_method]

In [ ]:
# define any arguments here except for 'seqs', which are defined later
kwargs = {
    'beam_size': 10,
    'beam_alpha': 0.3
}
num_samples: int = 100
scaled_radius: float = 1.0

In [ ]:
gen_smis_lst = []
for sml in tqdm(expert_smiles, total=len(expert_smiles)):
    gen_smis = chem_sample([sml], num_samples=num_samples, **kwargs)
    gen_smis_lst.append(gen_smis[0])

In [ ]:
flattened = list(itertools.chain.from_iterable(gen_smis_lst))
if len(flattened) == 0:
    raise ValueError("No valid molecules generated :( Please try again.")
len(flattened)

In [ ]:
def format_kwargs(kwargs: dict) -> str:
    return "_".join([f"{k}_{v}" for k, v in kwargs.items()])

In [ ]:
fine_tuned_str = "100_steps"
# save to .csv
pd.DataFrame(
    data={"SMILES": flattened}
).to_csv(f"data/outputs/bionemo_{expert_smiles_filename}_{fine_tuned_str}_num_samples_{num_samples}_sampling_method_{sampling_method}_scaled_radius_{scaled_radius}_{format_kwargs(kwargs)}.csv", index=False)

In [ ]:
for ori_smis, gen_smis in zip(expert_smiles, gen_smis_lst):
    mol = Chem.MolFromSmiles(ori_smis)
    display(Draw.MolToImage(mol, legends=f"Original {ori_smis}"))
    gen_mols = [Chem.MolFromSmiles(smi) for smi in gen_smis[:25]] # limit the number of molecules to display
    display(Draw.MolsToGridImage(gen_mols, molsPerRow=6, legends=[smi for smi in gen_smis]))
    print("=====================================================================")
    print("\n")